# Credit Risk Analysis

The objective of this analysis is to understand which are the factors that are more effecting the probability of having an high credit risk of a given company, using initially only the data of the explanatory variables of previous year to predict the credit risk of the consequent one (true values vs forecasts).

Then it can be added new explanatory variables 'delayed' e.g. leverage of two years, three years, (...) before.

In this specific notebook the original dataset will be imported, then some data cleansing will be applied in order to obtain a clearer dataset to work with in the following steps.

# Data Import and Brief Statistics

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly as px
import os
import pickle

In [2]:
xls_file = pd.ExcelFile(r'C:\Users\lenovo\Downloads\credit-risk.xlsx')
df1=pd.read_excel(xls_file , sheet_name = [name for name in xls_file.sheet_names 
                                            if name not in ('Description','Sheet8')] )
df = pd.concat(df1.values(), ignore_index=True)

In [6]:
pd.set_option('display.max_columns', None)

# Data Exploration

In [7]:
df.head()

,No,Company name,Turnover.2020,Turnover.2019,Turnover.2018,Turnover.2017,Turnover.2016,Turnover.2015,EBIT.2020,EBIT.2019,EBIT.2018,EBIT.2017,EBIT.2016,EBIT.2015,PLTax.2020,PLTax.2019,PLTax.2018,PLTax.2017,PLTax.2016,PLTax.2015,MScore.2020,MScore.2019,MScore.2018,MScore.2017,MScore.2016,MScore.2015,Region,Country,NACE code,Sector 1,Sector 2,Leverage.2020,Leverage.2019,Leverage.2018,Leverage.2017,Leverage.2016,Leverage.2015,ROE.2020,ROE.2019,ROE.2018,ROE.2017,ROE.2016,ROE.2015,TAsset.2020,TAsset.2019,TAsset.2018,TAsset.2017,TAsset.2016,TAsset.2015
0,1,LENDLEASE S.R.L.,29458,16716,9612,8097,7941.0,5600.0,-1556.0,-4540.0,623.0,-412.0,885.0,-1479.0,-1402.0,-4674.0,22.0,-360.0,368.0,-1140.0,CC,CC,CCC,C,BB,C,Milano,Italy,4200,Civil engineering,Capital Goods,14.33,9.90,56.77,64.15,21.46,48.07,-43.63,-180.22,8.24,-146.65,60.76,-471.72,49263,28268,15455,15992,13597.0,11659.0
1,2,PRICEWATERHOUSECOOPERS BUSINESS SERVICES SRL (...,16731,16403,16843,12241,9252.0,9515.0,1838.0,841.0,2738.0,-864.0,-2212.0,-3572.0,1600.0,700.0,2577.0,-900.0,-2187.0,-4591.0,A,BBB,BBB,CC,CCC,CCC,Milano,Italy,7022,Activities of head offices; management consult...,Commercial and professional services,1.86,2.45,2.92,5.65,2.90,1.29,27.60,14.30,61.42,-55.57,-127.29,-87.13,16550,16887,16468,10773,6697.0,8933.0
2,3,EVISO S.P.A.,48568,43039,34302,25791,19760.0,6941.0,1661.0,1464.0,976.0,495.0,162.0,224.0,1159.0,1047.0,779.0,267.0,63.0,123.0,BBB,BBB,BBB,BB,BB,BB,Cuneo,Italy,3514,"ELECTRICITY, GAS, STEAM AND AIR CONDITIONING S...",Utilities,3.59,3.49,4.44,7.69,12.54,9.39,39.38,48.89,57.52,42.73,20.34,44.62,13500,9620,7371,5432,4170.0,2862.0
3,4,CASA SERVICE MACHINE,47999,43484,43043,41682,51267.0,52584.0,416.0,255.0,-855.0,-23.0,426.0,969.0,236.0,107.0,-1002.0,-197.0,430.0,602.0,BB,B,CCC,B,BB,BB,Pas-de-Calais,France,4661,"Wholesale trade, except of motor vehicles and ...",Retailing,3.54,3.89,4.15,2.64,3.21,3.18,8.42,5.69,-17.24,0.71,2.89,6.45,24978,25032,25729,21632,25403.0,24941.0
4,5,PANFERTIL SPA,45948,47336,45626,48222,57074.0,62263.0,44.0,713.0,-672.0,-1091.0,97.0,987.0,3.0,48.0,-599.0,-821.0,-10.0,-1116.0,B,BB,CCC,CCC,B,B,Ravenna,Italy,4675,"Wholesale trade, except of motor vehicles and ...",Retailing,2.17,1.98,2.13,2.15,2.15,2.11,0.03,0.41,-5.17,-6.74,0.03,-8.19,36823,34659,36205,38423,41847.0,41323.0


In [8]:
df.tail()

,No,Company name,Turnover.2020,Turnover.2019,Turnover.2018,Turnover.2017,Turnover.2016,Turnover.2015,EBIT.2020,EBIT.2019,EBIT.2018,EBIT.2017,EBIT.2016,EBIT.2015,PLTax.2020,PLTax.2019,PLTax.2018,PLTax.2017,PLTax.2016,PLTax.2015,MScore.2020,MScore.2019,MScore.2018,MScore.2017,MScore.2016,MScore.2015,Region,Country,NACE code,Sector 1,Sector 2,Leverage.2020,Leverage.2019,Leverage.2018,Leverage.2017,Leverage.2016,Leverage.2015,ROE.2020,ROE.2019,ROE.2018,ROE.2017,ROE.2016,ROE.2015,TAsset.2020,TAsset.2019,TAsset.2018,TAsset.2017,TAsset.2016,TAsset.2015
121248,21250,ASTOR VILLAGE S.R.L.,3161,4635,4742,4499,4277.0,3650.0,985.0,1818.0,1790.0,1248.0,1154.0,800.0,739.0,1185.0,1283.0,875.0,744.0,494.0,AA,AA,AA,AA,A,A,Lecce,Italy,5510,Accommodation,Consumer Services,0.17,0.22,0.24,0.26,0.29,0.34,5.44,9.23,11.01,8.44,7.83,6.31,15935,15664,14438,13054,12243.0,11695.0
121249,21251,ODONE & SLOA S.R.L.,3161,2562,2559,2334,3692.0,2537.0,60.0,101.0,27.0,10.0,9.0,74.0,21.0,6.0,1.0,-4.0,-7.0,21.0,B,CCC,CCC,CCC,CCC,CCC,Latina,Italy,2825,Manufacture of machinery and equipment n.e.c.,Capital Goods,20.56,23.61,25.57,27.67,29.35,20.71,18.38,6.00,0.62,-4.80,-7.85,-12.84,2487,2317,2351,2521,2797.0,3152.0
121250,21252,GARRIDO MURO SOCIEDAD LIMITADA,3161,3146,2989,3101,2746.0,3154.0,260.0,13.0,48.0,41.0,49.0,94.0,191.0,19.0,25.0,41.0,37.0,72.0,A,A,A,BBB,BBB,A,La Rioja,Spain,1520,Manufacture of leather and related products,Consumer Durables and Apparel,0.45,0.17,0.28,0.40,0.37,0.33,10.86,1.23,1.88,3.10,2.90,5.62,2547,1855,1692,1843,1773.0,1699.0
121251,21253,CENTRO INGROSSO JOLLY S.R.L.,3161,2519,2290,2244,1761.0,1821.0,74.0,48.0,60.0,42.0,39.0,8.0,54.0,23.0,23.0,21.0,20.0,1.0,BB,BB,B,B,B,CCC,Salerno,Italy,4649,"Wholesale trade, except of motor vehicles and ...",Retailing,3.29,3.01,3.25,3.19,13.49,13.12,7.78,3.65,3.74,3.54,18.85,0.58,2961,2552,2604,2474,1546.0,1222.0
121252,21254,SALONES COMATEL SL,3161,4514,4435,4231,3908.0,2051.0,194.0,733.0,830.0,1120.0,1028.0,-144.0,52.0,541.0,656.0,910.0,1072.0,-157.0,A,AA,AA,AA,AA,B,Valencia,Spain,9200,Gambling and betting activities,Consumer Services,0.04,0.08,0.13,0.12,0.15,0.48,1.51,13.69,15.59,25.60,40.55,-9.98,3576,4259,4747,3993,3027.0,2333.0


In [9]:
print('columns:',list(df.columns))

columns: ['No', 'Company name', 'Turnover.2020', 'Turnover.2019', 'Turnover.2018', 'Turnover.2017', 'Turnover.2016', 'Turnover.2015', 'EBIT.2020', 'EBIT.2019', 'EBIT.2018', 'EBIT.2017', 'EBIT.2016', 'EBIT.2015', 'PLTax.2020', 'PLTax.2019', 'PLTax.2018', 'PLTax.2017', 'PLTax.2016', 'PLTax.2015', 'MScore.2020', 'MScore.2019', 'MScore.2018', 'MScore.2017', 'MScore.2016', 'MScore.2015', 'Region', 'Country', 'NACE code', 'Sector 1', 'Sector 2', 'Leverage.2020', 'Leverage.2019', 'Leverage.2018', 'Leverage.2017', 'Leverage.2016', 'Leverage.2015', 'ROE.2020', 'ROE.2019', 'ROE.2018', 'ROE.2017', 'ROE.2016', 'ROE.2015', 'TAsset.2020', 'TAsset.2019', 'TAsset.2018', 'TAsset.2017', 'TAsset.2016', 'TAsset.2015']


**Name:** name of the company

**Turnover:** an accounting concept that calculates how quickly a business conducts its operations.Most often, it is used to understand how quickly a company collects cash from accounts receivable or how fast the company sells its inventory

**EBIT:** Earnings Before Interest and Taxes (EBIT) is an indicator of a company's profitability

**PLTax:** Principal Lifetime income Tax is a new type of tax (proposal) that would tax a person/company based on their cumulative income over their lifetime up until the filing date

**MScore:** credit risk level of the company (from AAA to D, where D is the highest credit risk level reachable).

**Region:** city where the company resides

**Country:** worldwide country of the company

**NACE code:** is the European statistical classification of economic activities. NACE groups organizations according to their business activities

**Sector 1:** very detailed description of the company's business activities (e.g., activities of head offices - management consult; manufacture of leather and related products; wholesale trade - except of motor vehicles; ...)

**Sector 2:** more general sector membership (e.g., capital goods; energy; diversified finance; ...)

**Leverage:** a strategy that companies use to increase assets, cash flows, and returns, though it can also magnify losses.There are two main types of leverage: financial and operating.To increase financial leverage, a firm may borrow capital through issuing fixed-income securities or by borrowing money directly from a lender

**ROE:** Return on Equity (ROE) is the measure of a company’s annual return (net income) divided by the value of its total shareholders’ equity, expressed as a percentage.Alternatively, ROE can also be derived by dividing the firm’s dividend growth rate by its earnings retention rate

**TAsset:** Total Assets, most commonly used in the context of a corporation, are defined as the assets owned by the entity that has an economic value whose benefits can be derived in the future


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121253 entries, 0 to 121252
Data columns (total 49 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   No             121253 non-null  int64  
 1   Company name   121252 non-null  object 
 2   Turnover.2020  121253 non-null  int64  
 3   Turnover.2019  121253 non-null  int64  
 4   Turnover.2018  121253 non-null  int64  
 5   Turnover.2017  121253 non-null  int64  
 6   Turnover.2016  121176 non-null  float64
 7   Turnover.2015  121108 non-null  float64
 8   EBIT.2020      121249 non-null  float64
 9   EBIT.2019      121252 non-null  float64
 10  EBIT.2018      121252 non-null  float64
 11  EBIT.2017      121252 non-null  float64
 12  EBIT.2016      121207 non-null  float64
 13  EBIT.2015      121203 non-null  float64
 14  PLTax.2020     121251 non-null  float64
 15  PLTax.2019     121251 non-null  float64
 16  PLTax.2018     121251 non-null  float64
 17  PLTax.2017     121251 non-nul

In [10]:
df.isnull().sum() # checking null values and duplicates

No                 0
Company name       1
Turnover.2020      0
Turnover.2019      0
Turnover.2018      0
Turnover.2017      0
Turnover.2016     77
Turnover.2015    145
EBIT.2020          4
EBIT.2019          1
EBIT.2018          1
EBIT.2017          1
EBIT.2016         46
EBIT.2015         50
PLTax.2020         2
PLTax.2019         2
PLTax.2018         2
PLTax.2017         2
PLTax.2016        49
PLTax.2015        51
MScore.2020        0
MScore.2019        0
MScore.2018        0
MScore.2017        0
MScore.2016        0
MScore.2015        0
Region             1
Country            0
NACE code          0
Sector 1           0
Sector 2           0
Leverage.2020      0
Leverage.2019      0
Leverage.2018      0
Leverage.2017      0
Leverage.2016      0
Leverage.2015      0
ROE.2020           6
ROE.2019          15
ROE.2018          14
ROE.2017           7
ROE.2016          72
ROE.2015          73
TAsset.2020        0
TAsset.2019        0
TAsset.2018        0
TAsset.2017        0
TAsset.2016  

In [12]:
df.duplicated().sum()

np.int64(0)

In [26]:
df.dropna(inplace=True) # dropped null values  as there are very less null values it will not make much impact 

In [27]:
df.shape

(121008, 49)

In [28]:
df.drop(columns=['No'], inplace=True)

# Store the data in csv

In [ ]:
df.to_csv(r'C:\Users\lenovo\Downloads\credit-risk-cleaned.csv', index=False)